# l6e: Pipeline Budget Enforcement for AI Agents

`l6e` enforces a budget across your entire agent pipeline run — and lets your agent code read budget state and adapt mid-run.

This notebook covers four things:

1. **Automatic stage rerouting** — declare which stages can use cheaper models, l6e handles the rest
2. **Hard enforcement** — `BudgetMode.HALT` raises `BudgetExceeded` before a call goes out
3. **Agent-driven adaptation** — your agent reads `ctx.budget_status()` and decides what to do when budget runs low
4. **Running with a real API key** — drop-in with LiteLLM or any completion client

In [34]:
# !pip install 'l6e[langchain]' langchain-core litellm

---
## Act 0 — The rerouting table

A 3-stage pipeline, all requesting `gpt-4o`. One policy declaration, zero pipeline changes.

In [35]:
from __future__ import annotations

import l6e
from l6e.costs import LiteLLMCostEstimator

MODEL_FRONTIER = "gpt-4o"
MODEL_LOCAL    = "ollama/qwen2.5:7b"

estimator = LiteLLMCostEstimator()

STAGE_TOKENS: dict[str, tuple[int, int]] = {
    "retrieval":  (850, 120),
    "reasoning":  (620, 380),
    "formatting": (430, 95),
}

# Baseline: all three stages on gpt-4o, no l6e
baseline_cost = sum(estimator.estimate(MODEL_FRONTIER, pt, ct) for pt, ct in STAGE_TOKENS.values())

print("Baseline pipeline (all gpt-4o, no l6e)")
print("-" * 52)
for stage, (pt, ct) in STAGE_TOKENS.items():
    cost = estimator.estimate(MODEL_FRONTIER, pt, ct)
    print(f"  {stage:<14}  gpt-4o   {pt:>5}p + {ct:>4}c = ${cost:.5f}")
print("-" * 52)
print(f"  Total:         ${baseline_cost:.5f}")

Baseline pipeline (all gpt-4o, no l6e)
----------------------------------------------------
  retrieval       gpt-4o     850p +  120c = $0.00333
  reasoning       gpt-4o     620p +  380c = $0.00535
  formatting      gpt-4o     430p +   95c = $0.00202
----------------------------------------------------
  Total:         $0.01070


In [36]:
from typing import Any, Optional
from langchain_core.language_models.fake import FakeListLLM
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.outputs import LLMResult
from langchain_core.prompts import PromptTemplate


class TokenAwareFakeLLM(FakeListLLM):
    """FakeListLLM with realistic token counts and a real model name in serialization."""
    stage: str = "retrieval"
    model_name: str = MODEL_FRONTIER

    @property
    def _llm_type(self) -> str:
        return self.model_name

    @property
    def _identifying_params(self) -> dict[str, Any]:
        return {"model": self.model_name, **super()._identifying_params}

    def _generate(
        self,
        prompts: list[str],
        stop: Optional[list[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> LLMResult:
        result = super()._generate(prompts, stop=stop, run_manager=run_manager, **kwargs)
        pt, ct = STAGE_TOKENS.get(self.stage, (500, 150))
        result.llm_output = {"token_usage": {"prompt_tokens": pt, "completion_tokens": ct}}
        return result


class FakeLocalRouter:
    """Simulates Ollama being available — works without Ollama installed."""
    def best_local_model(self) -> str | None:
        return MODEL_LOCAL


# Internal imports used below to inject FakeLocalRouter into the pipeline.
# Acts 1 and 2 use l6e.pipeline() directly — no internals needed there.
from l6e._classify import PromptComplexityClassifier
from l6e._log import LocalRunLog
from l6e.adapters.langchain import L6eCallbackHandler
from l6e.gate import ConstraintGate
from l6e.store import InMemoryRunStore
from l6e.pipeline import PipelineContext

policy_hero = l6e.PipelinePolicy(
    budget=0.02,
    budget_mode=l6e.BudgetMode.REROUTE,
    stage_routing={
        "retrieval":  l6e.StageRoutingHint.LOCAL,
        "reasoning":  l6e.StageRoutingHint.CLOUD_FRONTIER,
        "formatting": l6e.StageRoutingHint.CLOUD_STANDARD,
    },
)

hero_store = InMemoryRunStore(run_id="hero-demo", policy=policy_hero, estimator=estimator)
ctx_hero = PipelineContext(
    run_id="hero-demo",
    policy=policy_hero,
    gate=ConstraintGate(policy=policy_hero, router=FakeLocalRouter()),
    store=hero_store,
    log=LocalRunLog(),
    classifier=PromptComplexityClassifier(),
    estimator=estimator,
)

retrieval_chain  = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["Retrieved: AI agents cost money."],             stage="retrieval")
reasoning_chain  = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["Analysis: cost scales with model tier."],       stage="reasoning")
formatting_chain = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["**Finding**: reroute summarisation to local."], stage="formatting")

handler = L6eCallbackHandler(ctx_hero)
cfg     = {"callbacks": [handler]}

with ctx_hero:
    r1 = retrieval_chain.with_config(tags=["l6e_stage:retrieval"]).invoke({"input": "What do AI agents cost?"}, config=cfg)
    r2 = reasoning_chain.with_config(tags=["l6e_stage:reasoning"]).invoke({"input": r1}, config=cfg)
    r3 = formatting_chain.with_config(tags=["l6e_stage:formatting"]).invoke({"input": r2}, config=cfg)

summary_hero = hero_store.to_summary()

print("Per-stage routing decisions")
print("-" * 74)
print(f"{'Stage':<14}  {'Requested':<12}  {'Used (l6e decision)':<28}  {'Cost':>9}  Rerouted")
print("-" * 74)
for r in summary_hero.records:
    marker = "YES ←" if r.rerouted else "-"
    print(f"{r.stage or '?':<14}  {r.model_requested:<12}  {r.model_used:<28}  ${r.cost_usd:.5f}  {marker}")
print("-" * 74)
print(f"{'Total':<14}  {'':12}  {'':28}  ${summary_hero.total_cost:.5f}")
print()
print(f"Baseline (all gpt-4o):  ${baseline_cost:.5f}")
print(f"With l6e:               ${summary_hero.total_cost:.5f}")
print(f"Savings:                ${summary_hero.savings_usd:.5f}  ({summary_hero.savings_usd / baseline_cost * 100:.1f}% reduction)")


l6e substituted `ollama/qwen2.5:7b` for the retrieval stage before the call went out — zero code changes to the pipeline itself. The 31.1% reduction comes from a single policy declaration.

**At scale:**

| Runs / day | Saved / day | Saved / month |
|------------|------------|---------------|
| 100        | $0.33       | ~$10           |
| 1,000      | $3.33       | ~$100          |
| 10,000     | $33         | ~$1,000 — with zero code changes |

---
## Act 1 — Hard enforcement: `BudgetMode.HALT` + `BudgetExceeded`

For production and multi-tenant pipelines, you need a guaranteed ceiling — not advisory rerouting, but a hard stop before a call goes out. Set `budget_mode=HALT` and l6e raises `BudgetExceeded` before executing the call.

```python
policy = l6e.PipelinePolicy(
    budget=0.001,
    budget_mode=l6e.BudgetMode.HALT,
)
```

The exception carries `spent`, `budget`, and `reason` — enough to log, alert, or return a graceful partial result.

In [37]:
def fake_completion(model: str, messages: list[dict]) -> object:
    class FakeUsage:
        prompt_tokens = 850
        completion_tokens = 120
    class FakeResponse:
        usage = FakeUsage()
    return FakeResponse()


policy_halt = l6e.PipelinePolicy(
    budget=0.001,
    budget_mode=l6e.BudgetMode.HALT,
    on_budget_exceeded=l6e.OnBudgetExceeded.RAISE,
)

ctx_halt = l6e.pipeline("halt-demo", policy_halt)

caught_exception = None

with ctx_halt:
    # First call: retrieval — $0.00333, already 3.3× over the $0.001 budget
    ctx_halt.call(
        fn=fake_completion,
        model="gpt-4o",
        messages=[{"role": "user", "content": "What do AI agents cost?"}],
        stage="retrieval",
    )
    print(f"After retrieval: spent ${ctx_halt.budget_status().spent_usd:.5f} of ${policy_halt.budget:.3f}")
    print()

    # Second call: reasoning — l6e blocks it before the network call
    try:
        ctx_halt.call(
            fn=fake_completion,
            model="gpt-4o",
            messages=[{"role": "user", "content": "Analyze cost drivers."}],
            stage="reasoning",
        )
    except l6e.BudgetExceeded as e:
        caught_exception = e
        print(f"Caught BudgetExceeded:")
        print(f"  {e}")
        print()
        print(f"  e.spent:  ${e.spent:.5f}")
        print(f"  e.budget: ${e.budget:.5f}")
        print(f"  e.reason: {e.reason}")

After retrieval: spent $0.00333 of $0.001

Caught BudgetExceeded:
  Budget exceeded: spent $0.003325 of $0.001000 budget. budget_pressure:halt

  e.spent:  $0.00333
  e.budget: $0.00100
  e.reason: budget_pressure:halt


The exception is raised **before** `fake_completion` is called — l6e checks the gate first. The pipeline's LLM client never saw the second request.

---
## Act 2 — Agent self-regulation: `ctx.budget_status()`

No gateway, no observability tool, and no framework gives the agent itself visibility into its own spend mid-run. `ctx.budget_status()` is the only primitive that does this.

The agent calls `ctx.budget_status()` at any point — zero tokens, just arithmetic — and branches on `budget_pressure`. This is different from HALT: the agent decides what to do, not the enforcement layer.

```python
status = ctx.budget_status()
if status.budget_pressure in ("high", "critical"):
    return partial_result   # agent chose to skip, not l6e
```

In [38]:
policy_adapt = l6e.PipelinePolicy(
    budget=0.004,          # $0.004 total — retrieval alone costs 83% of it
    budget_mode=l6e.BudgetMode.REROUTE,
)

ctx_adapt = l6e.pipeline("adapt-demo", policy_adapt)

def fake_retrieval(model: str, messages: list[dict]) -> object:
    class FakeUsage:
        prompt_tokens = 850
        completion_tokens = 120
    class FakeResponse:
        usage = FakeUsage()
        content = "Retrieved context: AI agent costs scale with model tier and call frequency."
    return FakeResponse()

def fake_deep_reasoning(model: str, messages: list[dict]) -> object:
    class FakeUsage:
        prompt_tokens = 620
        completion_tokens = 380
    class FakeResponse:
        usage = FakeUsage()
    return FakeResponse()

with ctx_adapt:
    # Run retrieval — $0.00333 spent, 83% of the $0.004 budget
    ctx_adapt.call(
        fn=fake_retrieval,
        model=MODEL_FRONTIER,
        messages=[{"role": "user", "content": "What do AI agents cost?"}],
        stage="retrieval",
    )
    retrieval_out = "Retrieved context: AI agent costs scale with model tier and call frequency."

    # Agent checks budget state before committing to the next call
    status = ctx_adapt.budget_status()

    print("After retrieval:")
    print(f"  Spent:           ${status.spent_usd:.5f} of ${status.budget_usd:.5f}  ({status.pct_used:.0f}% used)")
    print(f"  Remaining:       ${status.remaining_usd:.5f}")
    print(f"  Budget pressure: {status.budget_pressure}")
    print()

    if status.budget_pressure in ("high", "critical"):
        # Agent self-regulates: skips the expensive reasoning call entirely
        final_result = f"[Partial — budget pressure: {status.budget_pressure}] {retrieval_out}"
        print("Agent self-regulated: skipped deep reasoning")
        print("Returned partial result from retrieval stage instead")
    else:
        ctx_adapt.call(
            fn=fake_deep_reasoning,
            model=MODEL_FRONTIER,
            messages=[{"role": "user", "content": retrieval_out}],
            stage="reasoning",
        )
        final_result = "Full reasoning completed."
        print(f"Full reasoning completed. Spent: ${ctx_adapt.budget_status().spent_usd:.5f}")

    print()
    after = ctx_adapt.budget_status()
    print(f"Run completed cleanly. Final spend: ${after.spent_usd:.5f} / ${after.budget_usd:.5f}")
    print(f"Final result: {final_result}")

After retrieval:
  Spent:           $0.00333 of $0.00400  (83% used)
  Remaining:       $0.00067
  Budget pressure: high

Agent self-regulated: skipped deep reasoning
Returned partial result from retrieval stage instead

Run completed cleanly. Final spend: $0.00333 / $0.00400
Final result: [Partial — budget pressure: high] Retrieved context: AI agent costs scale with model tier and call frequency.


Budget pressure jumped from `low` (before any calls) to `high` after retrieval. The agent read that state and chose to return a partial result — no exception, no external enforcement, the agent decided.

This is the pattern for graceful degradation: give the agent the numbers, let it reason about trade-offs at each step.

---
## Act 3 — LangChain: `L6eCallbackHandler`

The only changes to an existing LangChain pipeline:
1. Create a `PipelinePolicy`.
2. Attach `L6eCallbackHandler(ctx)` to `config={"callbacks": [handler]}`.
3. Tag each chain with `l6e_stage:<name>`.

The pipeline logic is unchanged.

### Stage inference via `PromptComplexityClassifier`

The `with_config(tags=["l6e_stage:..."])` pattern is explicit but verbose. Stage names can also be inferred automatically — l6e's built-in classifier uses structural heuristics (no LLM, ~2ms) to classify prompts as `LOW / MEDIUM / HIGH` complexity:

In [39]:
from l6e._classify import PromptComplexityClassifier

classifier = PromptComplexityClassifier()

# Stage name alone short-circuits content analysis
print(classifier.classify("Summarize this document.", stage="retrieval"))    # LOW
print(classifier.classify("Analyze causal relationships.", stage="reasoning"))  # HIGH

# Content-based inference (no stage hint)
print(classifier.classify("Summarize this document.", stage=None))           # LOW
print(classifier.classify("Compare, evaluate, and synthesize the three approaches.", stage=None))  # HIGH

low
high
low
medium


The classifier is what feeds `CallRecord.prompt_complexity` — the field the Pro profiler's Task Classification Tuner uses to learn which stages in *your* pipeline can tolerate local models without quality regression.

In [40]:
def _make_lc_ctx(run_id: str, policy: l6e.PipelinePolicy):
    """Build a PipelineContext with FakeLocalRouter for hardware-independent reroute demos."""
    store = InMemoryRunStore(run_id=run_id, policy=policy, estimator=estimator)
    ctx = PipelineContext(
        run_id=run_id,
        policy=policy,
        gate=ConstraintGate(policy=policy, router=FakeLocalRouter()),
        store=store,
        log=LocalRunLog(),
        classifier=PromptComplexityClassifier(),
        estimator=estimator,
    )
    return ctx, store


policy_lc2 = l6e.PipelinePolicy(
    budget=0.02,
    budget_mode=l6e.BudgetMode.REROUTE,
    stage_routing={
        "retrieval":  l6e.StageRoutingHint.LOCAL,
        "reasoning":  l6e.StageRoutingHint.CLOUD_FRONTIER,
        "formatting": l6e.StageRoutingHint.CLOUD_STANDARD,
    },
)

ctx_lc2, store_lc2 = _make_lc_ctx("langchain-demo", policy_lc2)

retrieval_chain2  = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["Retrieved: AI agents cost money."],             stage="retrieval")
reasoning_chain2  = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["Analysis: cost scales with model tier."],       stage="reasoning")
formatting_chain2 = PromptTemplate.from_template("{input}") | TokenAwareFakeLLM(responses=["**Finding**: reroute summarisation to local."], stage="formatting")

handler2 = L6eCallbackHandler(ctx_lc2)
cfg2     = {"callbacks": [handler2]}

with ctx_lc2:
    r1 = retrieval_chain2.with_config(tags=["l6e_stage:retrieval"]).invoke({"input": "What do AI agents cost?"}, config=cfg2)
    r2 = reasoning_chain2.with_config(tags=["l6e_stage:reasoning"]).invoke({"input": r1}, config=cfg2)
    r3 = formatting_chain2.with_config(tags=["l6e_stage:formatting"]).invoke({"input": r2}, config=cfg2)

summary_lc2 = store_lc2.to_summary()

print("Per-stage routing decisions")
print("-" * 74)
print(f"{'Stage':<14}  {'Requested':<12}  {'Used (l6e decision)':<28}  {'Cost':>9}  Rerouted")
print("-" * 74)
for r in summary_lc2.records:
    marker = "YES ←" if r.rerouted else "-"
    print(f"{r.stage or '?':<14}  {r.model_requested:<12}  {r.model_used:<28}  ${r.cost_usd:.5f}  {marker}")
print("-" * 74)
print(f"{'Total':<14}  {'':12}  {'':28}  ${summary_lc2.total_cost:.5f}")
print()
print(f"Baseline (all gpt-4o):  ${baseline_cost:.5f}")
print(f"With l6e:               ${summary_lc2.total_cost:.5f}")
print(f"Savings:                ${summary_lc2.savings_usd:.5f}  ({summary_lc2.savings_usd / baseline_cost * 100:.1f}% reduction)")

Per-stage routing decisions
--------------------------------------------------------------------------
Stage           Requested     Used (l6e decision)                Cost  Rerouted
--------------------------------------------------------------------------
retrieval       gpt-4o        ollama/qwen2.5:7b             $0.00000  YES ←
reasoning       gpt-4o        gpt-4o                        $0.00535  -
formatting      gpt-4o        gpt-4o                        $0.00202  -
--------------------------------------------------------------------------
Total                                                       $0.00738

Baseline (all gpt-4o):  $0.01070
With l6e:               $0.00738
Savings:                $0.00333  (31.1% reduction)


---
## Act 4 — Running with a real API key

Set your `OPENAI_API_KEY` (or any key LiteLLM supports) and run the cell below. `l6e.pipeline()` handles all wiring — no fakes, no internal imports.

The policy routes all stages to cloud models by default. To also reroute `retrieval` to a local Ollama model, change `CLOUD_STANDARD` to `LOCAL` on the retrieval line — l6e will probe for a running Ollama instance automatically.

In [41]:
import os
import litellm

if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run this cell.")
else:
    policy_real = l6e.PipelinePolicy(
        budget=0.05,
        budget_mode=l6e.BudgetMode.REROUTE,
        stage_routing={
            "retrieval":  l6e.StageRoutingHint.CLOUD_STANDARD,   # swap to LOCAL if Ollama is running
            "reasoning":  l6e.StageRoutingHint.CLOUD_FRONTIER,
            "formatting": l6e.StageRoutingHint.CLOUD_STANDARD,
        },
    )

    with l6e.pipeline("real-run", policy_real) as ctx:
        response = ctx.call(
            fn=lambda model, msgs: litellm.completion(model=model, messages=msgs),
            model="gpt-4o",
            messages=[{"role": "user", "content": "What are the main cost drivers for AI agent pipelines?"}],
            stage="retrieval",
        )
        status = ctx.budget_status()
        print(f"Model used:       {response.model}")
        print(f"Tokens:           {response.usage.prompt_tokens}p + {response.usage.completion_tokens}c")
        print(f"Cost:             ${status.spent_usd:.5f}")
        print(f"Budget pressure:  {status.budget_pressure}")
        print(f"Remaining:        ${status.remaining_usd:.5f} of ${status.budget_usd:.2f}")

Set OPENAI_API_KEY to run this cell.


---
## The run log

Every `RunSummary` is appended to `.l6e/runs.jsonl` on context exit — automatically, with no extra code.

Each entry includes the full per-call breakdown: `model_requested`, `model_used`, `stage`, token counts, cost, and whether the call was rerouted. The file grows with every run.

In [42]:
import json
from pathlib import Path

log_path = Path(".l6e/runs.jsonl")

if log_path.exists():
    lines = log_path.read_text().strip().splitlines()
    print(f".l6e/runs.jsonl — {len(lines)} run(s) recorded")
    print()
    last = json.loads(lines[-1])
    print("Most recent run:")
    print(json.dumps(last, indent=2))
else:
    print("No run log found — run the pipeline cells above first.")

.l6e/runs.jsonl — 19 run(s) recorded

Most recent run:
{
  "run_id": "langchain-demo",
  "policy": {
    "budget": 0.02,
    "budget_mode": "reroute",
    "on_budget_exceeded": "raise",
    "fallback_result": null,
    "latency_sla": null,
    "reroute_threshold": 0.8,
    "stage_routing": {
      "retrieval": "local",
      "reasoning": "cloud_frontier",
      "formatting": "cloud_standard"
    },
    "stage_overrides": {}
  },
  "total_cost": 0.0073750000000000005,
  "calls_made": 3,
  "reroutes": 1,
  "savings_usd": 0.0033250000000000007,
  "records": [
    {
      "call_index": 0,
      "model_requested": "gpt-4o",
      "model_used": "ollama/qwen2.5:7b",
      "prompt_tokens": 850,
      "completion_tokens": 120,
      "cost_usd": 0,
      "rerouted": true,
      "elapsed_ms": 0.4575420171022415,
      "stage": "retrieval",
      "prompt_complexity": null,
      "is_multi_turn": false
    },
    {
      "call_index": 1,
      "model_requested": "gpt-4o",
      "model_used": "gpt-4